# Test Generation Pipeline

Notebook ini menguji pipeline generation KUHP RAG:
1. **Unit test** — parsing verifikasi, format konteks, sitasi (tanpa LLM)
2. **Integrasi** — retrieval + generation + verifikasi end-to-end (butuh Ollama)

Jalankan cell **Unit Test** dulu (cepat, tanpa model). Kalau lolos, lanjut ke **Integrasi**.

---
## 1. Unit Test (tanpa LLM)

In [1]:
import sys; sys.path.insert(0, "src")
from langchain_core.documents import Document
from generation import (
    format_context, extract_cited_pasal, parse_verification,
    VerificationResult, GenerationResult, format_answer,
    generate,
)
from generation_prompts import (
    generate_messages, verify_messages,
    PESAN_ABSTAIN, PESAN_TIDAK_ADA_KONTEKS,
)
print("Import OK")

Import OK


In [2]:
# --- Test format_context ---
docs = [
    Document(id="476", page_content="Setiap Orang yang mengambil suatu Barang...", metadata={"pasal": 476}),
    Document(id="477", page_content="Pencurian dengan kekerasan...", metadata={"pasal": 477}),
]
ctx = format_context(docs)
assert "[Pasal 476]" in ctx
assert "[Pasal 477]" in ctx
assert "Setiap Orang" in ctx
print("✅ format_context")
print(ctx)

✅ format_context
[Pasal 476]
Setiap Orang yang mengambil suatu Barang...

[Pasal 477]
Pencurian dengan kekerasan...


In [3]:
# --- Test extract_cited_pasal ---
assert extract_cited_pasal("Menurut [Pasal 476] dan [Pasal 477], ...") == [476, 477]
assert extract_cited_pasal("Tidak ada sitasi di sini") == []
assert extract_cited_pasal("[Pasal 1] [Pasal 1] duplikat") == [1]
assert extract_cited_pasal("[Pasal 300], [Pasal 100], [Pasal 200]") == [100, 200, 300]
print("✅ extract_cited_pasal")

✅ extract_cited_pasal


In [4]:
# --- Test parse_verification ---

# Kasus 1: semua didukung
v1 = parse_verification(
    "DIDUKUNG: ya\nKLAIM TIDAK DIDUKUNG: tidak ada\nPASAL YANG DIRUJUK: 476, 477"
)
assert v1.supported == "ya"
assert v1.is_supported == True
assert v1.cited_pasal == [476, 477]
assert v1.unsupported_claims == []
print("✅ parse_verification: semua didukung")

# Kasus 2: sebagian didukung
v2 = parse_verification(
    "DIDUKUNG: sebagian\nKLAIM TIDAK DIDUKUNG: Hukuman 10 tahun tidak disebutkan\nPASAL YANG DIRUJUK: 476"
)
assert v2.supported == "sebagian"
assert v2.is_supported == True
assert len(v2.unsupported_claims) == 1
print("✅ parse_verification: sebagian didukung")

# Kasus 3: tidak didukung
v3 = parse_verification(
    "DIDUKUNG: tidak\nKLAIM TIDAK DIDUKUNG: Semua klaim salah\nPASAL YANG DIRUJUK: -"
)
assert v3.supported == "tidak"
assert v3.is_supported == False
print("✅ parse_verification: tidak didukung")

# Kasus 4: output kosong / gagal parse → default "tidak" (aman)
v4 = parse_verification("")
assert v4.is_supported == False
print("✅ parse_verification: output kosong → default tidak")

✅ parse_verification: semua didukung
✅ parse_verification: sebagian didukung
✅ parse_verification: tidak didukung
✅ parse_verification: output kosong → default tidak


In [5]:
# --- Test generate() tanpa konteks → langsung abstain ---
result = generate("apa hukuman pencurian?", [], verify=False)
assert result.abstained == True
assert result.is_supported == False
assert PESAN_TIDAK_ADA_KONTEKS in result.answer
print("✅ generate tanpa konteks → abstain")
print(f"   answer: {result.answer[:80]}...")

✅ generate tanpa konteks → abstain
   answer: Tidak ditemukan pasal KUHP yang relevan dengan pertanyaan ini. Pastikan pertanya...


In [6]:
# --- Test prompt messages format ---
msgs = generate_messages("apa hukuman pencurian?", "[Pasal 476]\nSetiap Orang...")
assert len(msgs) == 2
assert msgs[0]["role"] == "system"
assert msgs[1]["role"] == "user"
assert "apa hukuman pencurian?" in msgs[1]["content"]
assert "[Pasal 476]" in msgs[1]["content"]
print("✅ generate_messages format")

vmsgs = verify_messages("Jawaban test", "[Pasal 476]\nSetiap Orang...")
assert len(vmsgs) == 2
assert vmsgs[0]["role"] == "system"
assert "Jawaban test" in vmsgs[1]["content"]
print("✅ verify_messages format")

✅ generate_messages format
✅ verify_messages format


In [7]:
# --- Test format_answer ---
result_ok = GenerationResult(
    query="apa hukuman pencurian?",
    answer="Menurut [Pasal 476], pencurian diancam pidana penjara paling lama 5 tahun.",
    cited_pasal=[476],
    is_supported=True,
    verification=VerificationResult(supported="ya", cited_pasal=[476]),
    context_docs=docs,
)
output = format_answer(result_ok)
assert "Pasal 476" in output
assert "📖 Dasar hukum" in output
print("✅ format_answer (supported)")
print(output)
print()

result_abstain = GenerationResult(
    query="test",
    answer="Mohon maaf...",
    cited_pasal=[],
    is_supported=False,
    verification=VerificationResult(supported="tidak"),
    context_docs=[],
    abstained=True,
)
output2 = format_answer(result_abstain)
assert "📖 Dasar hukum" not in output2  # abstain → tidak ada sitasi
print("✅ format_answer (abstained)")

✅ format_answer (supported)
Menurut [Pasal 476], pencurian diancam pidana penjara paling lama 5 tahun.

📖 Dasar hukum: Pasal 476

✅ format_answer (abstained)


In [8]:
print("\n" + "="*50)
print("Semua unit test LULUS ✅")
print("="*50)
print("\nLanjut ke bagian Integrasi (butuh Ollama running).")


Semua unit test LULUS ✅

Lanjut ke bagian Integrasi (butuh Ollama running).


---
## 2. Integrasi (butuh Ollama + model + data)

In [9]:
import torch  # harus paling atas (lihat catatan macOS)
import sys, time; sys.path.insert(0, "src")
from retrieval import load_resources, search
from generation import generate, format_answer, ask

print("Loading resources...")
t0 = time.time()
res = load_resources()
print(f"Loaded {res.n} pasal dalam {time.time()-t0:.1f} detik")

Loading resources...
Loaded 624 pasal dalam 0.1 detik


### 2a. Test retrieval → generation (langkah per langkah)

In [10]:
query = "apa hukuman untuk orang yang mencuri motor?"
print(f"Query: {query}\n")

# Step 1: Retrieval
print("--- Tahap Retrieval ---")
t0 = time.time()
docs = search(query, "hybrid", res, top_k=3, use_reranker=False, use_rewriting=True)
print(f"Ditemukan {len(docs)} pasal dalam {time.time()-t0:.1f} detik:")
for d in docs:
    print(f"  Pasal {d.metadata['pasal']} (skor: {d.metadata.get('score', '?'):.4f})")
    print(f"    {d.page_content[:100]}...")

# Step 2: Generation (tanpa verifikasi dulu)
print("\n--- Tahap Generation (tanpa verifikasi) ---")
t0 = time.time()
result = generate(query, docs, verify=False)
print(f"Selesai dalam {time.time()-t0:.1f} detik")
print(f"Sitasi: {result.cited_pasal}")
print(f"\nJawaban:\n{result.answer}")

Query: apa hukuman untuk orang yang mencuri motor?

--- Tahap Retrieval ---


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Ditemukan 3 pasal dalam 39.3 detik:
  Pasal 476 (skor: 0.0328)
    Pasal 476

Setiap Orang yang mengambil suatu Barang yang sebagian atau seluruhnya milik orang lain, ...
  Pasal 486 (skor: 0.0323)
    Pasal 486
Setiap Orang yang secara melawan hukum memiliki suatu Barang yang sebagian atau seluruhnya...
  Pasal 542 (skor: 0.0308)
    Pasal 542

Setiap Orang yang menggunakan Kapal untuk menahan atau melakukan Kekerasan atau Ancaman K...

--- Tahap Generation (tanpa verifikasi) ---
Selesai dalam 12.2 detik
Sitasi: [476]

Jawaban:
Berdasarkan pasal yang tersedia, tindakan mengambil barang milik orang lain dengan maksud untuk dimiliki secara melawan hukum dikategorikan sebagai pencurian [Pasal 476].

Hukuman bagi pelaku pencurian adalah:
*   Pidana penjara paling lama 5 (lima) tahun, atau
*   Pidana denda paling banyak kategori V [Pasal 476].


In [11]:
# Step 3: Generation DENGAN verifikasi
print("--- Tahap Generation (dengan verifikasi) ---")
t0 = time.time()
result_v = generate(query, docs, verify=True)
print(f"Selesai dalam {time.time()-t0:.1f} detik")
print(f"Didukung: {result_v.is_supported} ({result_v.verification.supported})")
print(f"Sitasi: {result_v.cited_pasal}")
print(f"Abstain: {result_v.abstained}")
if result_v.verification.unsupported_claims:
    print(f"Klaim tidak didukung: {result_v.verification.unsupported_claims}")
print(f"\n{format_answer(result_v)}")

--- Tahap Generation (dengan verifikasi) ---
Selesai dalam 18.7 detik
Didukung: True (ya)
Sitasi: [476]
Abstain: False

Berdasarkan pasal-pasal yang tersedia, tindakan mengambil motor (yang merupakan suatu barang) milik orang lain dengan maksud untuk memilikinya secara melawan hukum dikategorikan sebagai pencurian.

Hukuman bagi pelaku pencurian adalah:
*   Pidana penjara paling lama 5 (lima) tahun, atau
*   Pidana denda paling banyak kategori V [Pasal 476].

📖 Dasar hukum: Pasal 476


### 2b. Test `ask()` end-to-end

In [17]:
pertanyaan = [
    "apa hukuman untuk orang yang mencuri motor?",
    "kalau nipu orang lewat olshop bisa dipenjara berapa tahun?",
    "pasal 300",
]

for q in pertanyaan:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    t0 = time.time()
    r = ask(q, res=res, method="hybrid", top_k=3, use_rewriting=True, use_expansion=True, verify=True)
    dt = time.time() - t0
    print(f"⏱  {dt:.1f} detik | Didukung: {r.is_supported} | Pasal: {r.cited_pasal} | Abstain: {r.abstained}")
    print(f"\n{format_answer(r)}")


Q: apa hukuman untuk orang yang mencuri motor?
⏱  86.6 detik | Didukung: True | Pasal: [476] | Abstain: False

Berdasarkan pasal-pasal yang tersedia, tindakan mengambil motor (yang merupakan suatu barang) milik orang lain dengan maksud untuk memilikinya secara melawan hukum dikategorikan sebagai pencurian.

Hukuman bagi pelaku pencurian adalah:
*   Pidana penjara paling lama 5 (lima) tahun, atau
*   Pidana denda paling banyak kategori V [Pasal 476].

📖 Dasar hukum: Pasal 476

Q: kalau nipu orang lewat olshop bisa dipenjara berapa tahun?
⏱  59.0 detik | Didukung: True | Pasal: [492] | Abstain: False

Berdasarkan pasal-pasal yang diberikan, tindakan menipu orang lain (seperti dalam transaksi belanja daring/olshop) yang dilakukan dengan menggunakan tipu muslihat atau rangkaian kata bohong untuk menggerakkan orang agar menyerahkan suatu barang, dikategorikan sebagai penipuan.

Pelaku penipuan tersebut dapat dipidana dengan pidana penjara paling lama 4 (empat) tahun atau pidana denda palin

### 2c. Edge cases

In [18]:
# Test scope gate: pertanyaan bukan hukum
print("--- Scope gate: pertanyaan bukan hukum ---")
r = ask("1 tambah 1 berapa", res=res, use_scope_gate=True, verify=True)
print(f"Abstain: {r.abstained}")
print(f"Answer: {r.answer}")
assert r.abstained == True

--- Scope gate: pertanyaan bukan hukum ---
Abstain: True
Answer: Pertanyaan ini di luar cakupan chatbot KUHP.


In [19]:
# Test pasal tidak ada di korpus
print("--- Pasal tidak ditemukan ---")
r = ask("jelaskan pasal 9999", res=res, verify=True)
print(f"Abstain: {r.abstained}")
print(f"Answer: {r.answer}")

--- Pasal tidak ditemukan ---
Abstain: True
Answer: Tidak ditemukan pasal KUHP yang relevan dengan pertanyaan ini. Pastikan pertanyaan berkaitan dengan hukum pidana Indonesia (UU No. 1 Tahun 2023 tentang KUHP).


In [20]:
# Test lookup langsung (menyebut nomor pasal)
print("--- Lookup pasal langsung ---")
t0 = time.time()
r = ask("apa isi pasal 476?", res=res, verify=True)
print(f"⏱  {time.time()-t0:.1f} detik")
print(f"Didukung: {r.is_supported} | Pasal: {r.cited_pasal}")
print(f"\n{format_answer(r)}")

--- Lookup pasal langsung ---
⏱  14.9 detik
Didukung: True | Pasal: [476]

Isi Pasal 476 mengatur tentang tindak pidana pencurian sebagai berikut:

Setiap orang yang mengambil suatu barang yang sebagian atau seluruhnya milik orang lain, dengan maksud untuk dimiliki secara melawan hukum, dipidana karena pencurian [Pasal 476].

Sanksi bagi pelaku adalah:
*   Pidana penjara paling lama 5 (lima) tahun; atau
*   Pidana denda paling banyak kategori V [Pasal 476].

📖 Dasar hukum: Pasal 476


In [22]:
# --- Benchmark: waktu per tahap ---
import time

query = "apa hukuman untuk pencurian?"
docs = search(query, "hybrid", res, top_k=3, use_rewriting=True, use_expansion=True, use_reranker=True)

# Generate saja
t0 = time.time()
r1 = generate(query, docs, verify=False)
t_gen = time.time() - t0

# Generate + verify
t0 = time.time()
r2 = generate(query, docs, verify=True)
t_gen_v = time.time() - t0

print(f"Generate saja   : {t_gen:.1f} detik")
print(f"Generate+verify : {t_gen_v:.1f} detik")
print(f"Overhead verify : {t_gen_v - t_gen:.1f} detik")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Generate saja   : 15.7 detik
Generate+verify : 19.5 detik
Overhead verify : 3.8 detik
